# Compiled Fall Video Dataset — validation before modeling

This notebook validates the compiled Fall/No_Fall video and COCO-17 keypoint CSV dataset. It deliberately stops before Random Forest, RTMPose, MMAction2, or PoseC3D training.

**Rule:** a candidate pair is not considered verified until its keypoints align with its RGB video.

In [ ]:
from collections import Counter
from pathlib import Path
import re

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_columns', 50)
print('Imports complete.')

## 1. Configuration

The supplied path is used first. A standard Kaggle mount fallback is included because Kaggle commonly exposes attached datasets directly under `/kaggle/input/<slug>`.

In [ ]:
SUPPLIED_ROOT = Path('/kaggle/input/datasets/payutch/fall-video-dataset')
STANDARD_KAGGLE_ROOT = Path('/kaggle/input/fall-video-dataset')
DATASET_ROOT = SUPPLIED_ROOT if SUPPLIED_ROOT.exists() else STANDARD_KAGGLE_ROOT

VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.mpeg', '.mpg', '.m4v'}
EXPECTED_VIDEO_COUNT = 3140
EXPECTED_CSV_COUNT = 3140

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f'Neither {SUPPLIED_ROOT} nor {STANDARD_KAGGLE_ROOT} exists. '
        'Attach the Kaggle dataset and update DATASET_ROOT.'
    )
print('Using dataset root:', DATASET_ROOT)

## 2. Recursive inventory

In [ ]:
all_files = [path for path in DATASET_ROOT.rglob('*') if path.is_file()]
video_paths = sorted((p for p in all_files if p.suffix.lower() in VIDEO_EXTENSIONS), key=lambda p: str(p).lower())
csv_paths = sorted((p for p in all_files if p.suffix.lower() == '.csv'), key=lambda p: str(p).lower())

print(f'Videos: {len(video_paths):,} (expected {EXPECTED_VIDEO_COUNT:,})')
print(f'CSVs:   {len(csv_paths):,} (expected {EXPECTED_CSV_COUNT:,})')
print('Counts equal:', len(video_paths) == len(csv_paths))
display(pd.Series(Counter(p.suffix.lower() for p in video_paths), name='count').rename_axis('video_extension').reset_index())
display(pd.DataFrame({'first_video_paths': [str(p.relative_to(DATASET_ROOT)) for p in video_paths[:50]]}))
display(pd.DataFrame({'first_csv_paths': [str(p.relative_to(DATASET_ROOT)) for p in csv_paths[:50]]}))

In [ ]:
def folder_inventory(root: Path, max_depth: int = 4) -> pd.DataFrame:
    rows = []
    directories = sorted((p for p in root.rglob('*') if p.is_dir()), key=lambda p: str(p).lower())
    for directory in directories:
        relative = directory.relative_to(root)
        if len(relative.parts) <= max_depth:
            direct_files = [p for p in directory.iterdir() if p.is_file()]
            rows.append({
                'folder': str(relative),
                'direct_files': len(direct_files),
                'direct_videos': sum(p.suffix.lower() in VIDEO_EXTENSIONS for p in direct_files),
                'direct_csvs': sum(p.suffix.lower() == '.csv' for p in direct_files),
            })
    return pd.DataFrame(rows)

folder_df = folder_inventory(DATASET_ROOT)
display(folder_df)

## 3. Build file inventory and inspect pairing keys

Exact and normalized stem matches are trusted syntactically. Timestamp-nearest matches are only candidates and remain unverified.

In [ ]:
def binary_label_from_path(path: Path):
    parts = {part.lower().replace(' ', '_') for part in path.relative_to(DATASET_ROOT).parts}
    if 'fall' in parts:
        return 1
    if 'no_fall' in parts or 'nofall' in parts or 'no-fall' in parts:
        return 0
    return pd.NA

def normalized_stem(path: Path):
    stem = path.stem.lower()
    stem = re.sub(r'(?:[_\s-]*(?:keypoints?|pose|skeleton))+$', '', stem)
    return re.sub(r'[^a-z0-9]+', '', stem)

def timestamp_token(path: Path):
    match = re.search(r'(20\d{6})[^0-9]?(\d{6})', path.stem)
    if not match:
        return pd.NaT
    return pd.to_datetime(''.join(match.groups()), format='%Y%m%d%H%M%S', errors='coerce')

def parent_context(path: Path):
    parts = list(path.relative_to(DATASET_ROOT).parts[:-1])
    ignored = {'raw_video', 'raw video', 'keypoints_csv', 'keypoints csv', 'keypoint_csv'}
    return '/'.join(part.lower() for part in parts if part.lower() not in ignored)

def file_rows(paths, kind):
    return [{
        'kind': kind, 'path': str(p), 'relative_path': str(p.relative_to(DATASET_ROOT)),
        'filename': p.name, 'stem': p.stem, 'normalized_stem': normalized_stem(p),
        'timestamp': timestamp_token(p), 'parent_context': parent_context(p),
        'label': binary_label_from_path(p),
    } for p in paths]

video_df = pd.DataFrame(file_rows(video_paths, 'video'))
csv_df = pd.DataFrame(file_rows(csv_paths, 'csv'))
display(video_df.head())
display(csv_df.head())

In [ ]:
def unique_key_table(df: pd.DataFrame, key_columns: list[str], value_name: str):
    counts = df.groupby(key_columns, dropna=False).size().rename(f'{value_name}_key_count').reset_index()
    return df.merge(counts, on=key_columns, how='left')

pair_keys = ['label', 'parent_context', 'normalized_stem']
videos_keyed = unique_key_table(video_df, pair_keys, 'video')
csvs_keyed = unique_key_table(csv_df, pair_keys, 'csv')
exact_pairs = videos_keyed.merge(
    csvs_keyed, on=pair_keys, how='inner', suffixes=('_video', '_csv')
)
exact_pairs = exact_pairs.loc[(exact_pairs.video_key_count == 1) & (exact_pairs.csv_key_count == 1)].copy()
exact_pairs['pairing_method'] = 'unique_normalized_stem'
exact_pairs['pair_verified'] = False

matched_video_paths = set(exact_pairs.path_video)
matched_csv_paths = set(exact_pairs.path_csv)
unmatched_videos = video_df.loc[~video_df.path.isin(matched_video_paths)].copy()
unmatched_csvs = csv_df.loc[~csv_df.path.isin(matched_csv_paths)].copy()

print('Unique normalized-stem pairs:', len(exact_pairs))
print('Unmatched videos:', len(unmatched_videos))
print('Unmatched CSVs:', len(unmatched_csvs))
display(exact_pairs[['relative_path_video', 'relative_path_csv', 'pairing_method']].head(30))
display(unmatched_videos[['relative_path', 'normalized_stem', 'timestamp']].head(30))
display(unmatched_csvs[['relative_path', 'normalized_stem', 'timestamp']].head(30))

### Search CSVs for an embedded video reference

This reads only headers and a few initial rows. If an explicit source-video field exists, it should take priority over timestamp guessing.

In [ ]:
REFERENCE_COLUMN_PATTERN = re.compile(r'(video|file|source|clip|path)', re.IGNORECASE)
reference_findings = []
for csv_path in csv_paths[:min(200, len(csv_paths))]:
    sample = pd.read_csv(csv_path, nrows=5)
    candidates = [column for column in sample.columns if REFERENCE_COLUMN_PATTERN.search(str(column))]
    if candidates:
        reference_findings.append({
            'csv_path': str(csv_path.relative_to(DATASET_ROOT)),
            'candidate_columns': candidates,
            'values': {column: sample[column].dropna().astype(str).unique().tolist() for column in candidates},
        })
print(f'CSVs with possible video-reference columns among inspected files: {len(reference_findings)}')
display(pd.DataFrame(reference_findings).head(30))

### Diagnostic timestamp-nearest candidates

These candidates are produced only for inspection. They are not added to the verified pairing table. A candidate is shown only when video and CSV are mutual nearest neighbours within the same label/context.

In [ ]:
def mutual_nearest_timestamp_candidates(videos: pd.DataFrame, csvs: pd.DataFrame) -> pd.DataFrame:
    candidates = []
    group_columns = ['label', 'parent_context']
    for group_key, video_group in videos.dropna(subset=['timestamp']).groupby(group_columns, dropna=False):
        csv_group = csvs.dropna(subset=['timestamp'])
        for column, value in zip(group_columns, group_key if isinstance(group_key, tuple) else (group_key,)):
            csv_group = csv_group.loc[csv_group[column].eq(value)]
        if csv_group.empty:
            continue
        csv_times = csv_group.timestamp.astype('int64').to_numpy()
        video_times = video_group.timestamp.astype('int64').to_numpy()
        for video_position, video_row in enumerate(video_group.itertuples(index=False)):
            csv_position = int(np.argmin(np.abs(csv_times - video_row.timestamp.value)))
            reverse_video_position = int(np.argmin(np.abs(video_times - csv_times[csv_position])))
            if reverse_video_position == video_position:
                csv_row = csv_group.iloc[csv_position]
                candidates.append({
                    'video_path': video_row.path, 'csv_path': csv_row.path,
                    'video_relative_path': video_row.relative_path, 'csv_relative_path': csv_row.relative_path,
                    'time_difference_seconds': abs((video_row.timestamp - csv_row.timestamp).total_seconds()),
                    'label': video_row.label, 'parent_context': video_row.parent_context,
                    'pairing_method': 'mutual_nearest_timestamp_UNVERIFIED', 'pair_verified': False,
                })
    return pd.DataFrame(candidates)

timestamp_candidates = mutual_nearest_timestamp_candidates(unmatched_videos, unmatched_csvs)
print('Mutual timestamp-nearest candidates:', len(timestamp_candidates))
display(timestamp_candidates.head(50))
if not timestamp_candidates.empty:
    display(timestamp_candidates.time_difference_seconds.describe())

## 4. Infer source dataset conservatively

Source identity is inferred only from explicit path tokens. Generic timestamp filenames remain `Unknown`.

In [ ]:
SOURCE_PATTERNS = {
    'FallVision': [r'fall[\s_-]*vision'],
    '29SubjectDataset': [r'29[\s_-]*subject', r'2017[\s_-]*activit'],
    'MultipleCameras': [r'multiple[\s_-]*camera', r'multicam', r'camera[\s_-]*fall'],
}

def infer_source(relative_path: str):
    normalized = relative_path.lower()
    matches = [name for name, patterns in SOURCE_PATTERNS.items() if any(re.search(p, normalized) for p in patterns)]
    if len(matches) == 1:
        return matches[0], 'explicit path token'
    if len(matches) > 1:
        return 'Unknown', f'ambiguous: {matches}'
    return 'Unknown', 'no explicit source token'

source_records = video_df.relative_path.apply(infer_source)
video_df[['source_dataset', 'source_evidence']] = pd.DataFrame(source_records.tolist(), index=video_df.index)
display(video_df.groupby(['source_dataset', 'source_evidence'], dropna=False).size().rename('videos').reset_index())

## 5. Inspect CSV schemas across labels and folders

In [ ]:
CANONICAL_COCO17 = [
    'Nose', 'Left Eye', 'Right Eye', 'Left Ear', 'Right Ear',
    'Left Shoulder', 'Right Shoulder', 'Left Elbow', 'Right Elbow',
    'Left Wrist', 'Right Wrist', 'Left Hip', 'Right Hip',
    'Left Knee', 'Right Knee', 'Left Ankle', 'Right Ankle',
]

def normalize_column(column):
    return re.sub(r'[^a-z0-9]', '', str(column).lower())

COLUMN_ALIASES = {
    'frame': {'frame', 'frameid', 'framenumber', 'frameindex'},
    'keypoint': {'keypoint', 'keypointname', 'joint', 'jointname', 'bodypart'},
    'x': {'x', 'xcoordinate', 'keypointx'},
    'y': {'y', 'ycoordinate', 'keypointy'},
    'confidence': {'confidence', 'score', 'keypointscore', 'probability'},
}

def resolve_columns(columns):
    normalized = {normalize_column(column): column for column in columns}
    return {field: next((normalized[a] for a in aliases if a in normalized), None) for field, aliases in COLUMN_ALIASES.items()}

def evenly_sample_paths(paths, count):
    if len(paths) <= count:
        return list(paths)
    positions = np.linspace(0, len(paths) - 1, count, dtype=int)
    return [paths[position] for position in positions]

inspection_paths = []
for _, group in csv_df.groupby(['label', 'parent_context'], dropna=False):
    inspection_paths.extend(evenly_sample_paths([Path(p) for p in group.path], 3))
inspection_paths = list(dict.fromkeys(inspection_paths))
print('Representative CSVs selected:', len(inspection_paths))

In [ ]:
def inspect_csv(csv_path: Path):
    df = pd.read_csv(csv_path)
    resolved = resolve_columns(df.columns)
    result = {
        'csv_path': str(csv_path.relative_to(DATASET_ROOT)), 'rows': len(df),
        'columns': list(df.columns), 'resolved_columns': resolved,
        'missing_required_columns': [key for key, value in resolved.items() if value is None],
    }
    if all(resolved.values()):
        frame, keypoint, confidence = resolved['frame'], resolved['keypoint'], resolved['confidence']
        result.update({
            'unique_frames': df[frame].nunique(dropna=True),
            'unique_keypoints': sorted(df[keypoint].dropna().astype(str).unique().tolist()),
            'unique_keypoint_count': df[keypoint].nunique(dropna=True),
            'duplicate_frame_joint_rows': int(df.duplicated([frame, keypoint]).sum()),
            'missing_values': int(df[[frame, keypoint, resolved['x'], resolved['y'], confidence]].isna().sum().sum()),
            'confidence_min': pd.to_numeric(df[confidence], errors='coerce').min(),
            'confidence_mean': pd.to_numeric(df[confidence], errors='coerce').mean(),
            'confidence_max': pd.to_numeric(df[confidence], errors='coerce').max(),
        })
    return result

schema_inspection_df = pd.DataFrame(inspect_csv(path) for path in inspection_paths)
display(schema_inspection_df)

for path in inspection_paths[:min(6, len(inspection_paths))]:
    print('\n', path.relative_to(DATASET_ROOT))
    display(pd.read_csv(path).head(10))

## 6. COCO-17 consistency audit

This audits every CSV by default. If Kaggle memory is tight, set `AUDIT_ALL_CSVS = False` for an initial representative run.

In [ ]:
AUDIT_ALL_CSVS = True
CANONICAL_LOOKUP = {re.sub(r'[^a-z0-9]', '', name.lower()): name for name in CANONICAL_COCO17}

def canonical_joint_name(value):
    return CANONICAL_LOOKUP.get(re.sub(r'[^a-z0-9]', '', str(value).lower()))

def audit_coco17(csv_path: Path):
    df = pd.read_csv(csv_path)
    columns = resolve_columns(df.columns)
    base = {'csv_path': str(csv_path), 'schema_valid': all(columns.values())}
    if not base['schema_valid']:
        return {**base, 'complete_frame_rate': np.nan, 'unknown_joint_names': None}
    df = df.rename(columns={value: key for key, value in columns.items()})
    df['canonical_joint'] = df.keypoint.map(canonical_joint_name)
    unknown = sorted(df.loc[df.canonical_joint.isna(), 'keypoint'].dropna().astype(str).unique())
    duplicates = int(df.dropna(subset=['canonical_joint']).duplicated(['frame', 'canonical_joint']).sum())
    joints_per_frame = df.dropna(subset=['canonical_joint']).groupby('frame').canonical_joint.nunique()
    return {
        **base, 'num_frames': int(df.frame.nunique()),
        'complete_frames': int((joints_per_frame == 17).sum()),
        'incomplete_frames': int((joints_per_frame != 17).sum()),
        'complete_frame_rate': float((joints_per_frame == 17).mean()) if len(joints_per_frame) else 0.0,
        'duplicate_frame_joint_rows': duplicates, 'unknown_joint_names': unknown,
    }

audit_paths = csv_paths if AUDIT_ALL_CSVS else inspection_paths
coco_audit_df = pd.DataFrame(audit_coco17(path) for path in audit_paths)
print('Audited CSVs:', len(coco_audit_df))
display(coco_audit_df.describe(include='all'))
display(coco_audit_df.loc[(~coco_audit_df.schema_valid) | (coco_audit_df.complete_frame_rate < 1) | (coco_audit_df.duplicate_frame_joint_rows > 0)].head(50))

## 7. Reconstruct `T × 17 × 3` pose tensors

Missing coordinates remain `NaN`; missing confidences are zero. No interpolation is performed. Actual CSV frame identifiers are returned alongside the tensors.

In [ ]:
def reconstruct_pose_tensor(csv_path: Path):
    raw = pd.read_csv(csv_path)
    columns = resolve_columns(raw.columns)
    missing_columns = [field for field, column in columns.items() if column is None]
    if missing_columns:
        raise ValueError(f'{csv_path.name} is missing required fields: {missing_columns}')
    df = raw.rename(columns={column: field for field, column in columns.items()}).copy()
    df['canonical_joint'] = df.keypoint.map(canonical_joint_name)
    df = df.dropna(subset=['frame', 'canonical_joint'])
    if df.duplicated(['frame', 'canonical_joint']).any():
        duplicates = df.loc[df.duplicated(['frame', 'canonical_joint'], keep=False), ['frame', 'keypoint']]
        raise ValueError(f'Duplicate frame/joint rows found. Examples:\n{duplicates.head()}')
    frame_ids = np.sort(df.frame.unique())
    full_index = pd.MultiIndex.from_product([frame_ids, CANONICAL_COCO17], names=['frame', 'canonical_joint'])
    indexed = df.set_index(['frame', 'canonical_joint']).reindex(full_index)
    xy = indexed[['x', 'y']].apply(pd.to_numeric, errors='coerce').to_numpy(np.float32).reshape(len(frame_ids), 17, 2)
    score = pd.to_numeric(indexed.confidence, errors='coerce').fillna(0).to_numpy(np.float32).reshape(len(frame_ids), 17)
    combined = np.concatenate([xy, score[..., None]], axis=-1)
    diagnostics = {
        'frames': len(frame_ids), 'shape': combined.shape,
        'missing_joint_slots': int(np.isnan(xy).any(axis=2).sum()),
        'non_contiguous_frame_gaps': int(np.sum(np.diff(frame_ids.astype(float)) > 1)) if len(frame_ids) > 1 else 0,
    }
    return frame_ids, xy, score, combined, diagnostics

if not exact_pairs.empty:
    SELECTED_CSV_PATH = Path(exact_pairs.iloc[0].path_csv)
elif not timestamp_candidates.empty:
    SELECTED_CSV_PATH = Path(timestamp_candidates.iloc[0].csv_path)
else:
    SELECTED_CSV_PATH = csv_paths[0]

frame_ids, keypoint_xy, keypoint_score, pose_t17x3, tensor_diagnostics = reconstruct_pose_tensor(SELECTED_CSV_PATH)
print('Selected CSV:', SELECTED_CSV_PATH.relative_to(DATASET_ROOT))
print(tensor_diagnostics)
assert pose_t17x3.shape == (len(frame_ids), 17, 3)

## 8. Overlay pose on RGB video

Choose a syntactically matched pair or an explicitly inspected timestamp candidate. `CSV_TO_VIDEO_FRAME_OFFSET` handles 1-based versus 0-based frame numbering. Start with 0, then test `-1` or `+1` only if the overlays show a consistent one-frame shift.

In [ ]:
if not exact_pairs.empty:
    SELECTED_VIDEO_PATH = Path(exact_pairs.iloc[0].path_video)
    SELECTED_CSV_PATH = Path(exact_pairs.iloc[0].path_csv)
    SELECTED_PAIR_METHOD = exact_pairs.iloc[0].pairing_method
elif not timestamp_candidates.empty:
    SELECTED_VIDEO_PATH = Path(timestamp_candidates.iloc[0].video_path)
    SELECTED_CSV_PATH = Path(timestamp_candidates.iloc[0].csv_path)
    SELECTED_PAIR_METHOD = 'timestamp candidate — NOT VERIFIED'
else:
    raise RuntimeError('No pairing candidate exists. Do not visualize unrelated first files; inspect pairing evidence first.')

CSV_TO_VIDEO_FRAME_OFFSET = 0
print('Video:', SELECTED_VIDEO_PATH.relative_to(DATASET_ROOT))
print('CSV:  ', SELECTED_CSV_PATH.relative_to(DATASET_ROOT))
print('Method:', SELECTED_PAIR_METHOD)

In [ ]:
COCO17_EDGES = [
    (0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9),
    (6, 8), (8, 10), (5, 11), (6, 12), (11, 12), (11, 13),
    (13, 15), (12, 14), (14, 16),
]

def video_info(video_path: Path):
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f'Could not open video: {video_path}')
    info = {
        'video_frames': int(capture.get(cv2.CAP_PROP_FRAME_COUNT)),
        'fps': float(capture.get(cv2.CAP_PROP_FPS)),
        'width': int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
        'height': int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    capture.release()
    return info

def read_video_frame(video_path: Path, frame_index: int):
    capture = cv2.VideoCapture(str(video_path))
    capture.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    ok, frame_bgr = capture.read()
    capture.release()
    if not ok:
        raise RuntimeError(f'Could not decode frame {frame_index} from {video_path.name}')
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

def draw_coco17(axis, frame_rgb, xy, scores, confidence_threshold=0.25):
    axis.imshow(frame_rgb)
    finite = np.isfinite(xy).all(axis=1)
    visible = finite & (scores >= confidence_threshold)
    for start, end in COCO17_EDGES:
        if visible[start] and visible[end]:
            axis.plot([xy[start, 0], xy[end, 0]], [xy[start, 1], xy[end, 1]], color='lime', linewidth=2)
    axis.scatter(xy[visible, 0], xy[visible, 1], c='red', s=24, edgecolors='white', linewidths=0.6)
    axis.axis('off')

frame_ids, xy, scores, _, diagnostics = reconstruct_pose_tensor(SELECTED_CSV_PATH)
info = video_info(SELECTED_VIDEO_PATH)
print('Video info:', info)
print('Pose info:', diagnostics)
print('Coordinate ranges:', {'x_min': np.nanmin(xy[..., 0]), 'x_max': np.nanmax(xy[..., 0]), 'y_min': np.nanmin(xy[..., 1]), 'y_max': np.nanmax(xy[..., 1])})

sample_positions = np.unique(np.linspace(0, len(frame_ids) - 1, min(9, len(frame_ids)), dtype=int))
fig, axes = plt.subplots(3, 3, figsize=(17, 12))
axes = axes.reshape(-1)
for axis, pose_position in zip(axes, sample_positions):
    csv_frame_id = int(frame_ids[pose_position])
    video_frame_index = csv_frame_id + CSV_TO_VIDEO_FRAME_OFFSET
    if not 0 <= video_frame_index < info['video_frames']:
        axis.text(0.5, 0.5, f'Out of range\nCSV {csv_frame_id} → video {video_frame_index}', ha='center', va='center')
        axis.axis('off')
        continue
    frame_rgb = read_video_frame(SELECTED_VIDEO_PATH, video_frame_index)
    draw_coco17(axis, frame_rgb, xy[pose_position], scores[pose_position])
    axis.set_title(f'CSV frame {csv_frame_id} → video frame {video_frame_index}\nmean confidence {scores[pose_position].mean():.2f}')
for axis in axes[len(sample_positions):]:
    axis.axis('off')
plt.suptitle(f'{SELECTED_PAIR_METHOD} — visual verification required', fontsize=15)
plt.tight_layout()
plt.show()

## 9. Validation checkpoint — stop before modeling

Record the answers before continuing:

- Are there exactly 3140 videos and 3140 CSVs?
- Is pairing based on explicit identity, or merely timestamps?
- Do overlays prove that the selected video and CSV belong together?
- Are coordinates expressed in pixels and aligned to the decoded video dimensions?
- Are all joint names compatible with canonical COCO-17?
- What fraction of frames is incomplete or duplicated?
- Can source datasets be identified from real evidence?

If pairing or alignment fails, do **not** train. Export the diagnostic tables and inspect the dataset documentation or mapping metadata first.